# GSC-ViT — Salinas Spatial-Split Evaluation and Batch-128 Benchmark

This notebook evaluates an **existing GSC-ViT checkpoint** on the saved Salinas spatial split and benchmarks inference with batch size **128** to match the batch size recorded for the other models in the comparison table.

- It does **not** train or fine-tune the model.
- It does not overwrite the checkpoint.
- It saves test metrics, per-class metrics, confusion matrix, predictions, efficiency measurements, and run metadata under `results/gscvit/`.
- Accuracy and Macro-F1 are measured on the test split loaded from `outputs/salinas/salinas_spatial_split_seed42.npz`.
- If the checkpoint was trained with different preprocessing or a different split, results may not be comparable. Use the same model implementation and preprocessing used to produce the checkpoint.

In [1]:
from pathlib import Path
import sys, json, time, gc, platform, subprocess

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dfm.data.salinas import SalinasPatchDataset, load_salinas

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Device:", device)
print("PyTorch:", torch.__version__)

Project root: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup
Device: cuda
PyTorch: 2.5.1+cu118


In [2]:
# Paths: update CHECKPOINT_PATH if your .pth file is stored elsewhere.
SPLIT_PATH = PROJECT_ROOT / "outputs" / "salinas" / "salinas_spatial_split_seed42.npz"
CHECKPOINT_CANDIDATES = [
    PROJECT_ROOT / "results" / "gscvit" / "gscvit_salinas_best.pth",
    PROJECT_ROOT / "gscvit_salinas_best.pth",
    PROJECT_ROOT / "outputs" / "salinas" / "gscvit_salinas_best.pth",
]
CHECKPOINT_PATH = next((p for p in CHECKPOINT_CANDIDATES if p.exists()), None)

OUTPUT_DIR = PROJECT_ROOT / "results" / "gscvit" / "evaluation_batch128"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BENCH_BATCH_SIZE = 128
NUM_WARMUP = 20
NUM_TIMED = 100

if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"Spatial split file not found: {SPLIT_PATH}")
if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "Could not find gscvit_salinas_best.pth. Put it at "
        "'results/gscvit/gscvit_salinas_best.pth' or update CHECKPOINT_CANDIDATES."
    )

print("Split:", SPLIT_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("Output directory:", OUTPUT_DIR)
print("Benchmark batch size:", BENCH_BATCH_SIZE)

Split: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\salinas_spatial_split_seed42.npz
Checkpoint: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\results\gscvit\gscvit_salinas_best.pth
Output directory: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\results\gscvit\evaluation_batch128
Benchmark batch size: 128


In [3]:
# Load the exact saved spatial split.
split = np.load(SPLIT_PATH)
train_indices = split["train_indices"]
val_indices = split["val_indices"]
test_indices = split["test_indices"]

print(f"Train: {len(train_indices)} | Validation: {len(val_indices)} | Test: {len(test_indices)}")

data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"
scene = load_salinas(data_dir, download=True)
print("Cube shape (H, W, C):", scene.cube.shape)
print("Label map shape:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(scene.class_names))

test_base_dataset = SalinasPatchDataset(
    scene=scene,
    indices=test_indices,
    patch_size=15,
    normalize=True,
)

Train: 32337 | Validation: 10952 | Test: 10840
Cube shape (H, W, C): (512, 217, 204)
Label map shape: (512, 217)
Bands: 204
Classes: 16


In [4]:
class GSCViTDataset(Dataset):
    """Adapt Salinas 15x15 patches to the 16x16 GSC-ViT input interface."""
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        x, y = self.base_dataset[idx]
        if not torch.is_tensor(x):
            x = torch.from_numpy(x)
        x = x.float()
        if x.ndim != 3:
            raise ValueError(f"Expected [C,H,W], got {tuple(x.shape)}")
        if x.shape[-2:] == (15, 15):
            x = F.pad(x, (0, 1, 0, 1), mode="reflect")
        elif x.shape[-2:] != (16, 16):
            raise ValueError(f"Unexpected patch size: {tuple(x.shape)}")
        return x, torch.as_tensor(y).long()

test_dataset = GSCViTDataset(test_base_dataset)
test_loader = DataLoader(
    test_dataset,
    batch_size=BENCH_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)
print("Test dataset:", len(test_dataset), "| test loader batch size:", test_loader.batch_size)

Test dataset: 10840 | test loader batch size: 128


In [5]:
# Import the GSC-ViT implementation used by the original experiment.
import subprocess

gscvit_parent = PROJECT_ROOT / "external" / "gscvit"
repo_dir = gscvit_parent / "TGRS-GSC-VIT"
if not repo_dir.exists():
    gscvit_parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "https://github.com/flyzzie/TGRS-GSC-VIT.git", str(repo_dir)],
        check=True,
    )

if str(repo_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(repo_dir.resolve()))

from models.gscvit import gscvit

model = gscvit(dataset="sa").to(device)
print("Model implementation:", repo_dir / "models" / "gscvit.py")

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model implementation: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\external\gscvit\TGRS-GSC-VIT\models\gscvit.py


In [6]:
# Load checkpoint without training. Supports either a raw state_dict or common wrapper keys.
try:
    checkpoint_obj = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
except TypeError:  # compatibility with older PyTorch versions
    checkpoint_obj = torch.load(CHECKPOINT_PATH, map_location="cpu")

if isinstance(checkpoint_obj, dict):
    for key in ("state_dict", "model_state_dict", "model", "best_gscvit_state"):
        if key in checkpoint_obj and isinstance(checkpoint_obj[key], dict):
            checkpoint_obj = checkpoint_obj[key]
            break

if not isinstance(checkpoint_obj, dict):
    raise TypeError(f"Unsupported checkpoint object type: {type(checkpoint_obj)}")

# Remove a DataParallel prefix if present.
state_dict = {
    (k[7:] if k.startswith("module.") else k): v
    for k, v in checkpoint_obj.items()
}

load_result = model.load_state_dict(state_dict, strict=True)
model.eval()
print("Checkpoint loaded strictly:", CHECKPOINT_PATH)
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

Checkpoint loaded strictly: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\results\gscvit\gscvit_salinas_best.pth
Missing keys: []
Unexpected keys: []


In [7]:
@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    all_predictions, all_targets = [], []
    for x, y in loader:
        x = x.to(device, dtype=torch.float32, non_blocking=True)
        y = y.to(device, dtype=torch.long, non_blocking=True)
        logits = model(x)
        all_predictions.append(logits.argmax(dim=1).cpu().numpy())
        all_targets.append(y.cpu().numpy())
    y_pred = np.concatenate(all_predictions)
    y_true = np.concatenate(all_targets)
    return y_true, y_pred

y_true, y_pred = evaluate_model(model, test_loader, device)
test_accuracy = float(accuracy_score(y_true, y_pred))
test_macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
cm = confusion_matrix(y_true, y_pred)

print("GSC-ViT Salinas spatial-split test results")
print(f"Accuracy : {test_accuracy:.6f}")
print(f"Macro-F1 : {test_macro_f1:.6f}")

GSC-ViT Salinas spatial-split test results
Accuracy : 0.980627
Macro-F1 : 0.966688


In [8]:
# Save test metrics, class-wise report, confusion matrix, and predictions.
class_names = list(getattr(scene, "class_names", []))
labels = np.arange(len(class_names)) if class_names else None

report_dict = classification_report(
    y_true, y_pred, labels=labels, target_names=class_names if class_names else None,
    output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report_dict).transpose()
report_path = OUTPUT_DIR / "gscvit_test_classification_report.csv"
report_df.to_csv(report_path, index=True, index_label="class")

cm_df = pd.DataFrame(cm)
cm_path = OUTPUT_DIR / "gscvit_test_confusion_matrix.csv"
cm_df.to_csv(cm_path, index=False, header=False)

np.save(OUTPUT_DIR / "gscvit_test_y_true.npy", y_true)
np.save(OUTPUT_DIR / "gscvit_test_y_pred.npy", y_pred)

test_metrics = {
    "model": "GSC-ViT",
    "dataset": "Salinas",
    "split": "spatial",
    "split_file": str(SPLIT_PATH),
    "checkpoint": str(CHECKPOINT_PATH),
    "test_samples": int(len(y_true)),
    "accuracy": test_accuracy,
    "macro_f1": test_macro_f1,
    "evaluation_batch_size": int(BENCH_BATCH_SIZE),
    "device": str(device),
}
pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / "gscvit_test_metrics.csv", index=False)
with open(OUTPUT_DIR / "gscvit_test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved:")
for name in [
    "gscvit_test_metrics.csv", "gscvit_test_metrics.json",
    "gscvit_test_classification_report.csv", "gscvit_test_confusion_matrix.csv",
    "gscvit_test_y_true.npy", "gscvit_test_y_pred.npy"
]:
    print(" -", OUTPUT_DIR / name)

Saved:
 - c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\results\gscvit\evaluation_batch128\gscvit_test_metrics.csv
 - c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\results\gscvit\evaluation_batch128\gscvit_test_metrics.json
 - c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\results\gscvit\evaluation_batch128\gscvit_test_classification_report.csv
 - c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\results\gscvit\evaluation_batch128\gscvit_test_confusion_matrix.csv
 - c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\results\gscvit\evaluation_batch128\gscvit_test_y_true.npy
 - c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_back

In [9]:
# Efficiency benchmark at batch size 128.
# FLOPs are reported for the full benchmark batch and per sample where THOP is available.
def synchronize_if_cuda():
    if device.type == "cuda":
        torch.cuda.synchronize(device)

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
state_dict_bytes = sum(v.numel() * v.element_size() for v in model.state_dict().values())
model_size_mb = state_dict_bytes / (1024 ** 2)

# Use a full batch of 128 from the test set (or fail clearly if the test set is smaller).
if len(test_dataset) < BENCH_BATCH_SIZE:
    raise ValueError(
        f"Test split has {len(test_dataset)} samples, fewer than benchmark batch size {BENCH_BATCH_SIZE}."
    )
x_bench, _ = next(iter(DataLoader(
    test_dataset, batch_size=BENCH_BATCH_SIZE, shuffle=False, num_workers=0,
    pin_memory=(device.type == "cuda")
)))
x_bench = x_bench.to(device, dtype=torch.float32)

gflops_batch = None
gflops_per_sample = None
try:
    from thop import profile
    with torch.no_grad():
        flops, _ = profile(model, inputs=(x_bench,), verbose=False)
    gflops_batch = float(flops / 1e9)
    gflops_per_sample = float(flops / BENCH_BATCH_SIZE / 1e9)
except Exception as exc:
    print("THOP FLOPs measurement unavailable:", repr(exc))

model.eval()
with torch.no_grad():
    for _ in range(NUM_WARMUP):
        _ = model(x_bench)
synchronize_if_cuda()

times_ms = []
with torch.no_grad():
    for _ in range(NUM_TIMED):
        synchronize_if_cuda()
        start = time.perf_counter()
        _ = model(x_bench)
        synchronize_if_cuda()
        times_ms.append((time.perf_counter() - start) * 1000.0)

times_ms = np.asarray(times_ms, dtype=float)
latency_batch_ms = float(times_ms.mean())
latency_batch_median_ms = float(np.median(times_ms))
latency_batch_p95_ms = float(np.percentile(times_ms, 95))
latency_batch_std_ms = float(times_ms.std(ddof=1)) if len(times_ms) > 1 else 0.0
latency_per_sample_ms = latency_batch_ms / BENCH_BATCH_SIZE
throughput_samples_sec = BENCH_BATCH_SIZE / (latency_batch_ms / 1000.0)

peak_gpu_memory_mb = None
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)
    with torch.no_grad():
        for _ in range(10):
            _ = model(x_bench)
    synchronize_if_cuda()
    peak_gpu_memory_mb = float(torch.cuda.max_memory_allocated(device) / (1024 ** 2))

benchmark = {
    "model": "GSC-ViT",
    "dataset": "Salinas",
    "split": "spatial",
    "checkpoint": str(CHECKPOINT_PATH),
    "accuracy": test_accuracy,
    "macro_f1": test_macro_f1,
    "parameters": int(trainable_parameters),
    "parameters_m": float(trainable_parameters / 1e6),
    "model_size_mb": float(model_size_mb),
    "gflops_batch": gflops_batch,
    "gflops_per_sample": gflops_per_sample,
    "peak_gpu_memory_mb": peak_gpu_memory_mb,
    "batch_size": int(BENCH_BATCH_SIZE),
    "latency_batch_ms": latency_batch_ms,
    "latency_batch_median_ms": latency_batch_median_ms,
    "latency_batch_p95_ms": latency_batch_p95_ms,
    "latency_batch_std_ms": latency_batch_std_ms,
    "latency_per_sample_ms": float(latency_per_sample_ms),
    "throughput_samples_sec": float(throughput_samples_sec),
    "warmup_iterations": int(NUM_WARMUP),
    "timed_iterations": int(NUM_TIMED),
    "device": str(device),
}
benchmark_path = OUTPUT_DIR / "gscvit_efficiency_batch128.csv"
pd.DataFrame([benchmark]).to_csv(benchmark_path, index=False)
with open(OUTPUT_DIR / "gscvit_efficiency_batch128.json", "w", encoding="utf-8") as f:
    json.dump(benchmark, f, indent=2)

print("GSC-ViT batch-128 efficiency benchmark")
for key, value in benchmark.items():
    print(f"{key:28s}: {value}")
print("Saved:", benchmark_path)

GSC-ViT batch-128 efficiency benchmark
model                       : GSC-ViT
dataset                     : Salinas
split                       : spatial
checkpoint                  : c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\results\gscvit\gscvit_salinas_best.pth
accuracy                    : 0.9806273062730627
macro_f1                    : 0.9666879812641892
parameters                  : 179024
parameters_m                : 0.179024
model_size_mb               : 0.6898040771484375
gflops_batch                : 3.388903424
gflops_per_sample           : 0.026475808
peak_gpu_memory_mb          : 119.5908203125
batch_size                  : 128
latency_batch_ms            : 29.438163000013446
latency_batch_median_ms     : 28.47739999560872
latency_batch_p95_ms        : 31.848379997245495
latency_batch_std_ms        : 3.578605129587014
latency_per_sample_ms       : 0.22998564843760505
throughput_samples_sec      : 4348.097

In [10]:
# Save environment and reproducibility metadata for interpreting the benchmark.
metadata = {
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "pytorch_version": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_version": torch.version.cuda,
    "device": str(device),
    "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
    "checkpoint_path": str(CHECKPOINT_PATH),
    "checkpoint_size_bytes": int(CHECKPOINT_PATH.stat().st_size),
    "split_path": str(SPLIT_PATH),
    "split_seed_in_filename": 42,
    "patch_size_before_padding": 15,
    "model_input_spatial_size": "16x16",
    "padding_mode": "reflect",
    "benchmark_batch_size": BENCH_BATCH_SIZE,
    "warmup_iterations": NUM_WARMUP,
    "timed_iterations": NUM_TIMED,
}
with open(OUTPUT_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("All outputs saved to:", OUTPUT_DIR)

All outputs saved to: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\results\gscvit\evaluation_batch128


## Saved files

The notebook writes these files to `results/gscvit/evaluation_batch128/`:

- `gscvit_test_metrics.csv` and `.json`
- `gscvit_test_classification_report.csv`
- `gscvit_test_confusion_matrix.csv`
- `gscvit_test_y_true.npy` and `gscvit_test_y_pred.npy`
- `gscvit_efficiency_batch128.csv` and `.json`
- `run_metadata.json`

**Important:** batch size 128 here is the inference/efficiency benchmark batch size. It does not change how the checkpoint was trained. The earlier notebook's comparison table contains hard-coded values; do not treat those as newly reproduced measurements. Use the files generated by this notebook for updated claims.